# 🥈 Camada Silver — Cleansed

A camada Silver é responsável pela **limpeza, padronização e validação** dos dados vindos da Bronze. Aqui os dados se tornam confiáveis e tipados corretamente para uso analítico.

**Transformações aplicadas:**
- Remoção de registros com campos obrigatórios nulos (`equipment_id`, `production_qty`)
- Casting de tipos (`STRING` → `INT`, `TIMESTAMP`)
- Filtro de valores inválidos (`downtime_minutes < 0`)
- Padronização de campos categóricos (`UPPER`, `TRIM`)
- Adição de metadado de processamento (`processed_at`)

**Resultado esperado:** de 15 registros na Bronze, 12 chegam à Silver (3 removidos por falha de qualidade)

## 1. Criando o schema Silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

## 2. Transformação Bronze → Silver

Aplicação das regras de qualidade de dados:

| Regra | Campo | Ação |
|---|---|---|
| Campo obrigatório | `equipment_id` | Remove se nulo |
| Campo obrigatório | `production_qty` | Remove se nulo |
| Valor inválido | `downtime_minutes` | Remove se negativo |
| Padronização | `line_id`, `shift` | UPPER + TRIM |
| Tipagem | `production_qty`, `downtime_minutes`, `defect_qty` | Cast para INT |
| Tipagem | `event_timestamp` | Cast para TIMESTAMP |

In [0]:
%sql
CREATE OR REPLACE TABLE silver.production_events AS
SELECT
  equipment_id,
  UPPER(TRIM(line_id))  AS line_id,
  UPPER(TRIM(shift))    AS shift,
  CAST(event_timestamp  AS TIMESTAMP) AS event_timestamp,
  CAST(production_qty   AS INT)       AS production_qty,
  CAST(downtime_minutes AS INT)       AS downtime_minutes,
  CAST(defect_qty       AS INT)       AS defect_qty,
  source_system,
  ingested_at,
  current_timestamp() AS processed_at
FROM bronze.production_events
WHERE
  equipment_id     IS NOT NULL       -- remove equipment_id nulo
  AND production_qty  IS NOT NULL    -- remove production_qty nulo
  AND downtime_minutes >= 0          -- remove downtime negativo (valor inválido)
  AND production_qty   >= 0;         -- remove produção negativa

## 3. Validação — Comparativo Bronze vs Silver

Verificação do volume de registros removidos na etapa de limpeza.

In [0]:
%sql
-- Esperado: Bronze=15, Silver=12 (3 removidos por qualidade)
SELECT 'BRONZE' AS camada, COUNT(*) AS total FROM bronze.production_events
UNION ALL
SELECT 'SILVER' AS camada, COUNT(*) AS total FROM silver.production_events;

## 4. Visualização dos dados limpos

In [0]:
%sql
SELECT * FROM silver.production_events;